In [1]:
#%pip install xgboost

In [2]:
#%pip install tqdm

In [3]:
#%pip install audiomentations

In [4]:
#%pip install pyroomacoustics

In [5]:
#%pip install joblib

In [6]:
import os
import glob
import numpy as np
import librosa
import scipy.signal
import pandas as pd
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import kurtosis
from scipy.spatial.distance import cosine
from tqdm.notebook import tqdm  # Imports the Jupyter-friendly progress bar

In [7]:
def process_single_file(file_data):
    file_path, label = file_data
    
    try:
        features = build_advanced_feature_vector(file_path)
        
        return features, label, file_path
        
    except Exception as e:
        return None, label, file_path

In [11]:
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
import pickle


def preprocess_audio(audio_path, target_sr=16000):
    """Trims silence to prevent skewed statistical means."""
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    y_trimmed, _ = librosa.effects.trim(y, top_db=20)
    return y_trimmed, sr
    
def extract_glottal_residual_sharpness(y):
    """DEFENSE 1: LPC Residual Kurtosis"""
    y_pre = librosa.effects.preemphasis(y)
    lpc_coeffs = librosa.lpc(y_pre, order=16)
    residual = scipy.signal.lfilter(lpc_coeffs, [1.0], y_pre)
    return float(kurtosis(residual)), float(np.std(residual))

def extract_spectral_flux_dynamics(y):
    """DEFENSE 2: Spectral Flux Variance (The Consonant Punch)"""
    S = np.abs(librosa.stft(y))
    flux = np.maximum(0, np.diff(S, axis=1))
    total_flux_per_frame = np.sum(flux, axis=0)
    return float(np.std(total_flux_per_frame)), float(np.max(total_flux_per_frame)) 

def extract_modulation_energy(y, sr):
    """DEFENSE 3: Amplitude Modulation Spectrum (The Jaw Rhythm)"""
    analytic_signal = scipy.signal.hilbert(y)
    amplitude_envelope = np.abs(analytic_signal)
    env_fft = np.abs(np.fft.rfft(amplitude_envelope))
    freqs = np.fft.rfftfreq(len(amplitude_envelope), 1/sr)
    muscle_band = np.where((freqs >= 2) & (freqs <= 10))[0]
    
    if len(muscle_band) == 0:
        return 0.0
    return float(np.mean(env_fft[muscle_band]))

def extract_vocal_tract_formants(y, sr):
    """DEFENSE 4: MFCCs + Temporal Dynamics (Deltas) - FOR TIMBRE MATCH"""
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta_mfccs = librosa.feature.delta(mfccs)
    delta2_mfccs = librosa.feature.delta(mfccs, order=2)
    
    combined_mfccs = np.vstack([mfccs, delta_mfccs, delta2_mfccs])
    return np.hstack([np.mean(combined_mfccs, axis=1), np.std(combined_mfccs, axis=1)]) 

def extract_aliasing_signature(y, sr):
    """DEFENSE 5: High-Frequency Sibilance (RMS Aliasing Energy)"""
    stft_matrix = np.abs(librosa.stft(y))
    freqs = librosa.fft_frequencies(sr=sr)
    fold_back_indices = np.where((freqs >= 5000) & (freqs <= 8000))[0]
    band_rms_per_frame = np.sqrt(np.mean(stft_matrix[fold_back_indices, :]**2, axis=0))
    return float(np.mean(band_rms_per_frame)), float(np.std(band_rms_per_frame))

def extract_spectral_rolloff_variance(y, sr):
    """DEFENSE 6: Spectral Roll-off Variance (The Artificial Cutoff)"""
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.95)[0]
    return float(np.std(rolloff))

def extract_zcr_variance(y):
    """DEFENSE 7: Zero-Crossing Rate Variance (Micro-breathiness)"""
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    return float(np.std(zcr))
    
def build_advanced_feature_vector(audio_path):
    """Compiles the targeted feature array with MFCCs FIRST, then physics metrics."""
    y, sr = preprocess_audio(audio_path, target_sr=16000)
    
    # Extract Features
    glottal_kurtosis, glottal_std = extract_glottal_residual_sharpness(y)
    
    # Ignore the second return value (flux_max) by assigning it to '_'
    flux_std, _ = extract_spectral_flux_dynamics(y) 
    
    mod_energy = extract_modulation_energy(y, sr)
    rolloff_std = extract_spectral_rolloff_variance(y, sr)
    zcr_std = extract_zcr_variance(y)
    alias_mean, alias_std = extract_aliasing_signature(y, sr)
    
    formant_features = extract_vocal_tract_formants(y, sr)
    
    # Assemble Vector (MFCCs first, then Physics)
    feature_vector = np.hstack((
        formant_features,                          # Indices 0 to 119 (120 MFCC features)
        [glottal_kurtosis, glottal_std],           # Indices 120, 121
        [flux_std],                                # Index 122
        [mod_energy],                              # Index 123
        [rolloff_std],                             # Index 124
        [zcr_std],                                 # Index 125
        [alias_mean, alias_std]                    # Indices 126, 127
    ))
    
    return feature_vector

def extract_dataset(real_dir, fake_dir):
    """
    Loops through directories, extracts features, and assigns labels.
    Label 0 = Real, Label 1 = Fake (Spoof)
    """
    print("Starting dataset extraction...")
    features = []
    labels = []
    
    # Process Real Audio
    real_files = glob.glob(os.path.join(real_dir, "*.wav"))
    print(f"Found {len(real_files)} real audio files.")
    
    for file_path in tqdm(real_files, desc="Processing Real Audio", unit="file", colour="green"):
        try:
            vec = build_advanced_feature_vector(file_path)
            features.append(vec)
            labels.append(0)  # 0 = Genuine
        except Exception as e:
            print(f"\nError processing {file_path}: {e}")

    # Process Fake Audio
    fake_files = glob.glob(os.path.join(fake_dir, "*.wav"))
    print(f"Found {len(fake_files)} fake audio files.")
    
    # Notice the 'colour' spelling here!
    for file_path in tqdm(fake_files, desc="Processing Fake Audio", unit="file", colour="green"):
        try:
            vec = build_advanced_feature_vector(file_path)
            features.append(vec)
            labels.append(1)  # 1 = Spoof
        except Exception as e:
            print(f"\nError processing {file_path}: {e}")

    return np.array(features), np.array(labels)
    
def train_two_tower_models_xgb(X_full, y):
    """
    Splits the unified array into PUF/Physics and MFCCs, scales features, 
    and trains two separate XGBoost models.
    """
    print("\n--- Splitting Feature Arrays ---")
    
    # ARRAY SLICING (Updated for Future Expansion!)
    # 1. Isolate the MFCCs (The Biometric Identity Tower) - Fixed at exactly 120 features
    X_mfcc = X_full[:, 0:120]
    
    # 2. Isolate the Physics (The Anti-Spoofing PUF Tower) - Everything from index 120 onwards
    X_physics = X_full[:, 120:]

    # Split into Training and Testing sets (80% train, 20% test)
    X_mfcc_train, X_mfcc_test, y_train, y_test = train_test_split(X_mfcc, y, test_size=0.2, random_state=42)
    X_phys_train, X_phys_test, _, _ = train_test_split(X_physics, y, test_size=0.2, random_state=42)

    # --- 1. SCALE THE PHYSICS FEATURES ---
    scaler_phys = StandardScaler()
    X_phys_train_scaled = scaler_phys.fit_transform(X_phys_train)
    X_phys_test_scaled = scaler_phys.transform(X_phys_test)

    # --- TRAIN MODEL A: TIMBRE / BIOMETRICS (XGBoost) ---
    print("\n--- Training Model A: MFCC Biometric Classifier (XGBoost) ---")
    xgb_mfcc = XGBClassifier(
        n_estimators=200, 
        max_depth=6, 
        learning_rate=0.1, 
        random_state=42, 
        eval_metric='logloss'
    )
    xgb_mfcc.fit(X_mfcc_train, y_train)
    
    mfcc_preds = xgb_mfcc.predict(X_mfcc_test)
    print(f"MFCC Accuracy: {accuracy_score(y_test, mfcc_preds) * 100:.2f}%\n")
    print(classification_report(y_test, mfcc_preds, target_names=["Real", "Fake"]))

    # --- TRAIN MODEL B: PHYSICS / ANTI-SPOOFING (XGBoost) ---
    print("\n--- Training Model B: Physics PUF Classifier (XGBoost) ---")
    xgb_phys = XGBClassifier(
        n_estimators=400,        
        max_depth=6,             
        learning_rate=0.05,      
        subsample=0.8,           
        colsample_bytree=0.8,    
        random_state=42,
        eval_metric='logloss'
    )
    xgb_phys.fit(X_phys_train_scaled, y_train)
    
    # ---------------------------------------------------------
    # THRESHOLD LOGIC
    # ---------------------------------------------------------
    SECURITY_THRESHOLD = 0.50  
    phys_probs_real = xgb_phys.predict_proba(X_phys_test_scaled)[:, 0]
    phys_preds = (phys_probs_real < SECURITY_THRESHOLD).astype(int)

    print(f"\nPhysics Accuracy (at {SECURITY_THRESHOLD} threshold): {accuracy_score(y_test, phys_preds) * 100:.2f}%\n")
    print(classification_report(y_test, phys_preds, target_names=["Real", "Fake"]))

    # Return both models AND the scaler
    return xgb_mfcc, xgb_phys, scaler_phys

import os
import glob
import pickle
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# --- EXECUTION BLOCK ---
if __name__ == "__main__":
    REAL_AUDIO_DIR = "SPARC-audio-clips/resyn/gt5k/" 
    FAKE_AUDIO_DIR = "SPARC-audio-clips/resyn/resyn4k/"
    
    # ==========================================
    # 1. GATHER FILES
    # ==========================================
    print("Gathering audio files...")
    real_files = glob.glob(os.path.join(REAL_AUDIO_DIR, "*.wav"))
    fake_files = glob.glob(os.path.join(FAKE_AUDIO_DIR, "*.wav"))
    
    all_file_paths = real_files + fake_files
    all_labels = [0] * len(real_files) + [1] * len(fake_files) 
    tasks = list(zip(all_file_paths, all_labels))
    
    if len(tasks) == 0:
        print("❌ No data found. Check your folder paths.")
    else:
        print(f"Found {len(real_files)} Real and {len(fake_files)} Fake files.")
        
        # ==========================================
        # 2. EXTRACT DATA (JOBLIB MULTIPROCESSING)
        # ==========================================
        safe_cores = min(8, max(1, os.cpu_count() // 2))
        print(f"🚀 Firing up Joblib across {safe_cores} CPU cores...\n")
        
        # Joblib handles the heavy lifting cleanly
        # batch_size="auto" prevents memory leaks
        results = Parallel(n_jobs=safe_cores, batch_size="auto")(
            delayed(process_single_file)(task) for task in tqdm(tasks, desc="Extracting Features", ncols=100)
        )
        
        # Unpack the results
        X_features = []
        y_labels_list = []
        failed_files = []
        
        for features, label, file_path in results:
            if features is not None:
                X_features.append(features)
                y_labels_list.append(label)
            else:
                failed_files.append(file_path)
                    
        X_full = np.array(X_features)
        y_labels = np.array(y_labels_list)
        
        if failed_files:
            print(f"\n⚠️ Skipped {len(failed_files)} files due to errors.")
            
        # ==========================================
        # 3. TRAIN AND SAVE XGBOOST MODELS
        # ==========================================
        if len(X_full) > 0:
            print("\n--- Training XGBoost Models ---")
            model_mfcc, model_physics, scaler_physics = train_two_tower_models_xgb(X_full, y_labels)
            
            print("\n--- Saving Models to Disk ---")
            with open('tower1_mfcc_xgb.pkl', 'wb') as f:
                pickle.dump(model_mfcc, f)
                
            with open('tower2_physics_xgb.pkl', 'wb') as f:
                pickle.dump(model_physics, f)
                
            with open('physics_scaler.pkl', 'wb') as f:
                pickle.dump(scaler_physics, f)
                
            print("✅ XGBoost Models and Scaler successfully saved as .pkl files!")

            # ==========================================
            # 4. DISPLAY FEATURE IMPORTANCES
            # ==========================================
            print("\n--- Physics Model Feature Importances ---")
            physics_feature_names = [
                "Glottal Kurtosis", "Glottal Std", "Spectral Flux Std", 
                "Modulation Energy", "Rolloff Std", "ZCR Std",
                "Aliasing Mean", "Aliasing Std"
            ]
            
            importances = model_physics.feature_importances_
            importance_dict = dict(zip(physics_feature_names, importances))
            sorted_importances = sorted(importance_dict.items(), key=lambda item: item[1], reverse=True)
            
            for name, score in sorted_importances:
                print(f"{name:<20}: {score:.4f} ({score*100:.1f}%)")

Gathering audio files...
Found 5000 Real and 5000 Fake files.
🚀 Firing up Joblib across 8 CPU cores...



Extracting Features: 100%|███████████████████████████████████| 10000/10000 [01:39<00:00, 100.19it/s]



--- Training XGBoost Models ---

--- Splitting Feature Arrays ---

--- Training Model A: MFCC Biometric Classifier (XGBoost) ---
MFCC Accuracy: 99.65%

              precision    recall  f1-score   support

        Real       1.00      0.99      1.00      1012
        Fake       0.99      1.00      1.00       988

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000


--- Training Model B: Physics PUF Classifier (XGBoost) ---

Physics Accuracy (at 0.5 threshold): 91.70%

              precision    recall  f1-score   support

        Real       0.94      0.89      0.92      1012
        Fake       0.90      0.94      0.92       988

    accuracy                           0.92      2000
   macro avg       0.92      0.92      0.92      2000
weighted avg       0.92      0.92      0.92      2000


--- Saving Models to Disk ---
✅ XGBoost Models and Scaler successfully saved as .pkl f